# Extract voltage features (Week 3)

Reads `cycles_interpolated` from raw JSON and computes **ΔV(Q)** features (project plan §5.3 / Severson-style):

**ΔV(Q) = V_late(Q) − V_early(Q)** on a common discharge-capacity grid, with stats: mean, std, var, min, max.

Cycle pairs: **10→50** and **10→100** (early-cycle windows for later ablations).

Raw per-cycle voltage mean/min/max are **not** exported — in this dataset, `cycles_interpolated` uses a fixed 2.8–3.5 V grid, so those stats are identical for every cell.

Output: `data/processed/voltage_features.csv` (one row per kept cell).

Set `NUM_FILES = 2` for a quick test; `None` for all 134 cells (~5–10 min).

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    """Find project root even if the kernel cwd drifted after earlier chdir calls."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(
        f'Could not find data/raw/ starting from {here}. '
        'Restart the kernel, then run this cell first.'
    )


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from dedupe_policy import is_kept_file

NUM_FILES = None  # set to 2 for a quick test; None = all kept files
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'voltage_features.csv'
N_Q_GRID = 1000
DELTA_PAIRS = ((10, 50), (10, 100))  # plan: ΔV(Q); pairs align with 50/100-cycle windows


def fmt_seconds(seconds):
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, secs = divmod(int(seconds), 60)
    return f"{minutes}m {secs}s"


OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

files = sorted(ROOT.glob('data/raw/FastCharge*.json'))
files = [str(f) for f in files if is_kept_file(f.name)]
if NUM_FILES is not None:
    files = files[:NUM_FILES]

print('Project root:', ROOT)
print(f"Processing {len(files)} kept file(s)")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
def discharge_qv(curves, cycle_index):
    """Discharge V(Q) for one cycle (step_type == 'discharge')."""
    ci = np.asarray(curves['cycle_index'])
    st = np.asarray(curves['step_type'])
    mask = (ci == cycle_index) & (st == 'discharge')
    q = np.asarray(curves['discharge_capacity'], dtype=float)[mask]
    v = np.asarray(curves['voltage'], dtype=float)[mask]
    ok = np.isfinite(q) & np.isfinite(v)
    q, v = q[ok], v[ok]
    if len(q) < 2:
        return None, None
    order = np.argsort(q)
    return q[order], v[order]


def delta_v_stats(curves, cycle_early, cycle_late, n_grid=N_Q_GRID):
    """ΔV(Q) = V_late(Q) - V_early(Q) on overlapping Q range."""
    q_early, v_early = discharge_qv(curves, cycle_early)
    q_late, v_late = discharge_qv(curves, cycle_late)
    if q_early is None or q_late is None:
        return None

    q_hi = min(q_early.max(), q_late.max())
    q_lo = max(q_early.min(), q_late.min())
    if q_hi <= q_lo:
        return None

    grid = np.linspace(q_lo, q_hi, n_grid)
    v_early_i = np.interp(grid, q_early, v_early)
    v_late_i = np.interp(grid, q_late, v_late)
    delta_v = v_late_i - v_early_i

    return {
        'mean': float(np.mean(delta_v)),
        'std': float(np.std(delta_v)),
        'var': float(np.var(delta_v)),
        'min': float(np.min(delta_v)),
        'max': float(np.max(delta_v)),
    }


def extract_voltage_features(curves):
    features = {}
    stat_names = ('mean', 'std', 'var', 'min', 'max')

    for cycle_early, cycle_late in DELTA_PAIRS:
        stats = delta_v_stats(curves, cycle_early, cycle_late)
        for stat in stat_names:
            key = f'delta_v_{stat}_c{cycle_early}_c{cycle_late}'
            features[key] = stats[stat] if stats else np.nan

    return features

In [ ]:
if not files:
    raise FileNotFoundError(
        'No JSON files found. Re-run the setup cell above '
        '(if you ran it twice before fixing paths, restart the kernel first).'
    )

rows = []
loop_start = time.perf_counter()

for i, path in enumerate(files):
    file_id = os.path.basename(path)
    file_start = time.perf_counter()

    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    curves = data.get('cycles_interpolated')
    if curves is None:
        raise ValueError(f"Missing cycles_interpolated in {file_id}")

    row = {
        'file_id': file_id,
        'cell_id': data['barcode'],
        **extract_voltage_features(curves),
    }
    rows.append(row)

    file_elapsed = time.perf_counter() - file_start
    total_elapsed = time.perf_counter() - loop_start
    done = i + 1
    avg_per_file = total_elapsed / done
    remaining = len(files) - done
    eta = avg_per_file * remaining

    if done <= 3 or done % 10 == 0 or done == len(files):
        print(
            f"[{done}/{len(files)}] {file_id} | "
            f"this file: {fmt_seconds(file_elapsed)} | "
            f"running total: {fmt_seconds(total_elapsed)} | "
            f"ETA: {fmt_seconds(eta) if remaining else '0s'}"
        )

process_elapsed = time.perf_counter() - loop_start
voltage_features = pd.DataFrame(rows)
feature_cols = [c for c in voltage_features.columns if c not in ('file_id', 'cell_id')]

print()
print(f"Processed {len(voltage_features)} cells in {fmt_seconds(process_elapsed)}")
print(f"Feature columns ({len(feature_cols)}):", feature_cols)

In [ ]:
required = {'file_id', 'cell_id'}
missing = required - set(voltage_features.columns)
if missing:
    raise RuntimeError(
        f'Missing columns {missing}. Run the extraction cell above first '
        '(kernel → Restart & Run All is safest).'
    )

print('Rows:', len(voltage_features))
print('Unique file_id:', voltage_features['file_id'].nunique())
print('Duplicate file_id:', voltage_features['file_id'].duplicated().sum())
print()
print('Missing values per feature:')
print(voltage_features[feature_cols].isna().sum().sort_values(ascending=False))
print()
voltage_features.head()

In [ ]:
voltage_features.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}: {len(voltage_features)} rows, {len(feature_cols)} feature columns")